# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 0. 제출 정보
- 이름: 송누리
- GitHub ID: HauserNR
- 작성일: 2026-10-07
- 최종 제출 URL: https://github.com/HauserNR/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb

## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [32]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: /Users/kjj/MyLLMGit/llm-data-analysis-study/.venv/bin/python
현재 작업 폴더: /Users/kjj/MyLLMGit/llm-data-analysis-study/chapter04
프로젝트 루트: /Users/kjj/MyLLMGit/llm-data-analysis-study
데이터 폴더: /Users/kjj/MyLLMGit/llm-data-analysis-study/data/raw
결과 저장 폴더: /Users/kjj/MyLLMGit/llm-data-analysis-study/reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [33]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [34]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 1. 질문과 필요한 데이터 선택
### 내가 확인하려는 질문

어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?

### 사용한 파일/컬럼

금액 계산에는 order_items.csv의 quantity와 unit_price를 사용했다.
이 파일에는 주문 상태와 카테고리가 없어서 orders.csv의 order_status와 products.csv의 category를 연결했다.
주문은 order_id, 상품은 product_id로 연결했다. order_item_id는 상세 행이 중복되지 않았는지 확인할 때 사용했다.
상품별 결과에는 product_name을, 월별 결과에는 order_date를 사용했다.
고객별로는 customer_id를 기준으로 집계한 뒤 customers.csv에서 city만 가져왔다.
추가 실습에서는 고객의 gender·age, 상품의 price, 주문의 payment_method도 사용했다.

### 결과 관찰

CSV를 읽어 보니 고객은 150행, 상품은 100행, 주문은 300행, 주문 상세는 764행이었다.
주문 상태는 completed 184건, cancelled 64건, refunded 52건으로 나뉘어 있었다.

### 나의 해석과 판단

한 주문에 상품이 여러 개 들어갈 수 있으므로 주문 수와 상세 행 수를 구분했다.
이번 질문은 완료 주문 기준이기 때문에 cancelled와 refunded는 제외했다.
금액도 상품 목록의 price가 아니라 주문 상세에 기록된 unit_price로 계산했다.
연령이나 도시로 고객을 제한하면 전체 완료 주문에 대한 결과가 아니게 되므로, 그 조건은 필터 연습에만 사용했다.

### 업무·분석적 의미

이렇게 집계하면 완료 주문 금액에서 어느 상품군의 비중이 큰지 알 수 있다.
카테고리별 상품 운영을 검토할 때 먼저 살펴볼 대상을 정하는 데 쓸 수 있다.

### 한계와 추가 확인 사항

가상 쇼핑몰 데이터이므로 실제 기업의 실적을 설명하는 자료는 아니다.
원가와 할인, 세금, 배송비도 없어서 이 계산만으로 순매출이나 이익을 알 수는 없다.

### Evidence
![필터와 파생 컬럼](images/step01_question.png)

## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [35]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [36]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
42,43,뷰티 상품 043,뷰티,197000
57,58,식품 상품 058,식품,197000
23,24,스포츠 상품 024,스포츠,196000
36,37,뷰티 상품 037,뷰티,193000
8,9,스포츠 상품 009,스포츠,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
78,79,김종수,M,69,부산,2026-05-12
14,15,서민준,M,69,서울,2024-11-04
8,9,우영미,M,69,서울,2025-05-13
54,55,오민지,M,69,광주,2024-08-06
136,137,장진호,M,68,울산,2024-11-23
45,46,장현지,M,67,대전,2024-01-01
132,133,김민수,M,67,성남,2025-09-18
123,124,김성진,M,67,대구,2025-09-29
57,58,성도현,F,66,성남,2024-04-06
11,12,김동현,F,66,대전,2024-07-16


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [37]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건: order_status == "completed". 연령·도시 조건은 별도 연습에만 사용했다.
- 정렬 기준: 카테고리·상품·고객은 total_sales가 큰 순서로, 월별 결과는 날짜순으로 정렬했다.
- 만든 파생 컬럼: line_total, order_month, sales_ratio, average_order_value, customer_label.
- `line_total` 계산식: quantity × unit_price.

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰

첫 상세 행은 수량이 3, 단가가 102,000이어서 line_total이 306,000으로 계산됐다.
전체 주문 상세 금액은 255,610,000이었다. completed만 남기니 주문 184건, 상세 474행이 남았고 금액은 148,990,000이었다.
날짜 변환에 실패한 행은 없었다.

### 나의 해석과 판단

completed 조건을 빼면 금액이 106,620,000 더 커진다.
계산식이 같아도 어떤 주문을 포함하느냐에 따라 결과가 달라지므로, 이후 집계에서도 같은 필터를 유지했다.

### 업무·분석적 의미

전체 주문 금액과 완료 주문 금액을 따로 보면 취소·환불 주문이 섞인 수치를 구분할 수 있다.
결과를 전달할 때도 어떤 상태까지 포함했는지 함께 적어야 금액을 잘못 이해하지 않을 수 있다.

### 한계와 추가 확인 사항

전체 금액과 완료 주문 금액의 차이는 cancelled 62,181,000과 refunded 44,439,000을 더한 값과 같았다.
여기서 계산한 것은 제외한 주문의 상품 금액이다. 실제로 환불한 돈이나 손실액을 계산한 것은 아니다.

## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [38]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 3. merge 검증
- 병합한 데이터: order_items에 orders를 연결하고, 완료 주문 상세에 products를 연결했다. 고객별 집계에는 customers의 city를 붙였다.
- 사용한 key: 주문은 order_id, 상품은 product_id, 고객은 customer_id.
- `validate` 결과: 주문·상품 병합은 many_to_one, 고객 속성 병합은 one_to_one을 통과했다. 오른쪽 key의 중복은 모두 0건이었다.
- `indicator` 결과: both는 주문 764행, 상품 474행, 고객 100행이었다. left_only와 right_only는 모두 0행이었다.
- 병합 전/후 행 수: 주문 764→764행, 상품 474→474행, 고객 100→100행.

![merge 검증](images/step03_merge.png)

### 결과 관찰

세 번의 병합 모두 행 수가 그대로였고 연결되지 않은 행도 없었다.
병합 후에도 order_item_id는 중복되지 않았다. 상세가 없는 주문도 없었다.

### 나의 해석과 판단

상세 테이블에는 같은 order_id가 여러 번 나왔지만 orders에서는 중복이 없었다.
한 주문에 여러 상품이 들어 있는 구조이므로 상세 쪽의 반복은 그대로 두고, orders 쪽의 key가 고유한지 확인했다.
행 수와 금액도 유지되어 병합 때문에 같은 상세가 늘어나거나 빠지지는 않았다고 판단했다.

### 업무·분석적 의미

주문 테이블의 key가 중복되면 같은 상품 금액이 여러 번 더해질 수 있다.
반대로 연결되지 않은 상세를 버리면 금액이 빠진다. 병합 결과를 확인해야 이런 계산 오류를 찾을 수 있다.

### 한계와 추가 확인 사항

ID가 연결됐다고 해서 상품 정보까지 정확하다는 뜻은 아니다.
예를 들어 카테고리가 잘못 입력돼 있어도 총합은 같을 수 있다. 카테고리별 결과를 판단하려면 상품 정보도 정확해야 한다.

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [39]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 9. 상품 데이터 병합과 검증


In [40]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 10. 카테고리별·상품별 매출


In [41]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [42]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-09,3529000,5,705800.0
1,2025-10,16197000,16,1012312.0
2,2025-11,11694000,17,687882.0
3,2025-12,12319000,14,879929.0
4,2026-01,5144000,8,643000.0
5,2026-02,10034000,12,836167.0
6,2026-03,14254000,18,791889.0
7,2026-04,16634000,20,831700.0
8,2026-05,14367000,19,756158.0
9,2026-06,8943000,9,993667.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [43]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 4. completed 주문 범위와 집계
- 분석 범위 정의: 완료 주문 184건의 상세 474행. 해당 주문에서 구매된 상품은 98개, 고객은 100명이었다.
- 카테고리별 결과: 스포츠가 total_sales 31,743,000, total_quantity 295개로 가장 높았다. 전체 금액의 21.31%였다.
- 상품별 결과: 스포츠 상품 041이 total_sales 5,705,000, total_quantity 35개로 금액 1위였다.
- 월별 결과: 2026년 4월이 total_sales 16,634,000, order_count 20건으로 금액 1위였다.
- 고객별 결과: Customer 117이 total_sales 4,100,000, order_count 5건으로 금액 1위였다.

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰

카테고리별 금액은 스포츠가 가장 컸고 전자기기가 26,400,000으로 두 번째였다. 패션은 10,587,000으로 가장 작았다.
4월과 7월은 주문 수가 20건으로 같았지만 금액은 각각 16,634,000과 14,639,000이었다.
금액이 높은 고객 10명의 합계는 34,694,000으로 전체의 23.29%를 차지했다.

### 나의 해석과 판단

처음 질문에 대한 답은 스포츠였다. 수량도 295개로 가장 많았다.
다만 금액은 수량과 단가를 곱한 값이므로 많이 팔린 영향과 가격의 영향을 나눠 봐야 한다.
월별 결과에서도 주문 수가 같아도 금액이 달랐다. 주문 건수만으로는 구매 규모를 충분히 설명할 수 없었다.

### 업무·분석적 의미

스포츠 상품 중 어떤 상품이 금액을 많이 차지했는지 살펴보면 상품 운영을 검토할 대상을 좁힐 수 있다.
월별로는 주문 수와 주문당 금액을 함께 보고, 고객별로는 재구매 기록을 추가해 구매 패턴을 살펴볼 수 있다.

### 한계와 추가 확인 사항

total_sales는 완료 주문의 quantity × unit_price를 더한 값이다. 할인·세금·배송비 등의 처리를 확인하지 않았으므로 회계상 순매출이라고 부를 수는 없다.
원가가 없어 이익도 알 수 없고, 금액이 가장 크다는 이유만으로 가장 인기 있는 카테고리라고 단정할 수도 없다.
처음과 마지막 달인 2025년 9월과 2026년 9월은 한 달 전체가 포함되지 않아 다른 달과 단순 비교하기 어렵다.

## 13. 결과 저장하기


In [45]:
# 병합 전 원본 상세에서 completed 주문 금액 계산
completed_order_ids = orders.loc[
    orders["order_status"] == "completed", "order_id"
]

raw_completed_items = order_items.loc[
    order_items["order_id"].isin(completed_order_ids)
]

original_completed_total = (
    raw_completed_items["quantity"] * raw_completed_items["unit_price"]
).sum()

print("원본 completed 금액:", f"{original_completed_total:,}")


# 집계 결과 저장
outputs = {
    "ch04_category_sales.csv": category_sales,
    "ch04_product_sales.csv": product_sales,
    "ch04_monthly_sales.csv": monthly_summary,
    "ch04_customer_sales.csv": customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(filename, path.exists(), path.stat().st_size)


# 저장한 CSV를 다시 읽어 원본 집계표와 비교
for filename, original_df in outputs.items():
    loaded_df = pd.read_csv(REPORT_DIR / filename)

    pd.testing.assert_frame_equal(
        loaded_df,
        original_df.reset_index(drop=True),
        check_dtype=False,
    )

    assert loaded_df["total_sales"].sum() == original_completed_total

    print(f"\n{filename}: PASS")
    print("행·열 수:", loaded_df.shape)
    print("컬럼:", loaded_df.columns.tolist())
    print("금액 합계:", f"{loaded_df['total_sales'].sum():,}")

원본 completed 금액: 148,990,000
ch04_category_sales.csv True 246
ch04_product_sales.csv True 4310
ch04_monthly_sales.csv True 430
ch04_customer_sales.csv True 3286

ch04_category_sales.csv: PASS
행·열 수: (7, 4)
컬럼: ['category', 'total_quantity', 'total_sales', 'sales_ratio']
금액 합계: 148,990,000

ch04_product_sales.csv: PASS
행·열 수: (98, 5)
컬럼: ['product_id', 'product_name', 'category', 'total_quantity', 'total_sales']
금액 합계: 148,990,000

ch04_monthly_sales.csv: PASS
행·열 수: (13, 4)
컬럼: ['order_month', 'total_sales', 'order_count', 'average_order_value']
금액 합계: 148,990,000

ch04_customer_sales.csv: PASS
행·열 수: (100, 5)
컬럼: ['customer_id', 'order_count', 'total_sales', 'city', 'customer_label']
금액 합계: 148,990,000


In [29]:
completed_order_ids = orders.loc[
    orders["order_status"] == "completed", "order_id"
]

raw_completed_items = order_items.loc[
    order_items["order_id"].isin(completed_order_ids)
]

original_completed_total = (
    raw_completed_items["quantity"] * raw_completed_items["unit_price"]
).sum()

total_checks = pd.DataFrame([
    {
        "scope": "원본 completed line_total",
        "total_sales": original_completed_total,
    },
    {
        "scope": "상품 병합 후 completed",
        "total_sales": completed_sales_items["line_total"].sum(),
    },
    {
        "scope": "카테고리별 전체",
        "total_sales": category_sales["total_sales"].sum(),
    },
    {
        "scope": "상품별 전체",
        "total_sales": product_sales["total_sales"].sum(),
    },
    {
        "scope": "월별 전체",
        "total_sales": monthly_summary["total_sales"].sum(),
    },
    {
        "scope": "고객별 전체",
        "total_sales": customer_sales["total_sales"].sum(),
    },
])

total_checks["difference"] = (
    total_checks["total_sales"] - original_completed_total
)

total_checks["result"] = (
    total_checks["difference"]
    .eq(0)
    .map({True: "PASS", False: "FAIL"})
)

display(total_checks)

assert total_checks["difference"].eq(0).all(), (
    "집계 총합 불일치: 해석 전에 원인 확인"
)

assert (
    category_sales["total_quantity"].sum()
    == completed_sales_items["quantity"].sum()
)

assert (
    product_sales["total_quantity"].sum()
    == completed_sales_items["quantity"].sum()
)

print("PASS: 모든 금액 합계 =", f"{original_completed_total:,}")

,scope,total_sales,difference,result
0,원본 completed line_total,148990000,0,PASS
1,상품 병합 후 completed,148990000,0,PASS
2,카테고리별 전체,148990000,0,PASS
3,상품별 전체,148990000,0,PASS
4,월별 전체,148990000,0,PASS
5,고객별 전체,148990000,0,PASS


PASS: 모든 금액 합계 = 148,990,000


## 5. 총합 일치 검증
- 원본 completed `line_total` 합계: 148,990,000.
- 카테고리 합계: 148,990,000.
- 월별 합계: 148,990,000.
- 고객별 합계: 148,990,000.
- 차이 여부: 모든 합계가 같았고 원본 대비 차이는 0이었다.

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단

집계 기준만 바꾼 것이므로 포함된 주문이 같다면 총합도 같아야 한다.
원본 상세에서 completed 주문 ID에 해당하는 행만 골라 기준 금액을 구했다.
카테고리·상품·월·고객별 합계가 모두 148,990,000으로 같았다. 상품별 합계는 화면에 보인 상위 10개가 아니라 전체 98개를 사용했다.

합계가 다르면 먼저 completed 필터가 같은지 보고, 병합 후 행 수와 key 중복, 미매칭을 확인할 것이다.
그다음 그룹을 나누는 컬럼의 결측치, 날짜 변환 실패, 일부 결과만 더했는지, 수량과 단가 계산을 확인할 것이다.
월별·고객별 주문 수를 더한 값도 각각 184건이었다. 상품·카테고리별로는 한 주문이 여러 그룹에 들어갈 수 있으므로 주문 수를 단순히 더하면 안 된다.
총합이 같아도 원본의 수량이나 단가가 틀렸다면 계산 결과 역시 틀릴 수 있다는 점은 남는다.

## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


In [30]:
import pandas as pd
from IPython.display import display

# 1. 실제 컬럼과 주문 상태 확인
print("\n[1] 실제 데이터 구조 확인")

for name, df in [
    ("order_items", order_items),
    ("orders", orders),
    ("products", products),
]:
    print(f"{name}: {df.columns.tolist()}")

display(
    orders["order_status"]
    .value_counts(dropna=False)
    .rename("order_count")
    .to_frame()
)

# 2. LLM이 제안한 코드의 오류 확인
print("\n[2] 기존 LLM 코드 실행")
print("order_items.groupby('category')['line_total'].sum()")

try:
    display(order_items.groupby("category")["line_total"].sum())
except KeyError as exc:
    print("오류:", exc)
    print("category는 order_items가 아니라 products에 있다.")
    print("주문 상태도 orders에서 연결해야 completed만 선택할 수 있다.")

# 3. 원본 상세에서 금액 계산
print("\n[3] line_total = quantity × unit_price")

llm_items = order_items.copy()
llm_items["line_total"] = (
    llm_items["quantity"] * llm_items["unit_price"]
)

display(
    llm_items[
        ["order_id", "product_id", "quantity", "unit_price", "line_total"]
    ].head()
)

# 4. 주문 상태 연결과 병합 검증
print("\n[4] orders 병합과 검증")

print("orders.order_id 결측:", orders["order_id"].isna().sum())
print("orders.order_id 중복:", orders["order_id"].duplicated().sum())

assert orders["order_id"].notna().all()
assert orders["order_id"].is_unique

llm_orders = llm_items.merge(
    orders[["order_id", "order_status", "order_date"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

llm_orders_counts = llm_orders["_merge"].value_counts()

print("병합 전 행 수:", len(llm_items))
print("병합 후 행 수:", len(llm_orders))
display(llm_orders_counts.rename("rows").to_frame())

assert len(llm_orders) == len(llm_items)
assert llm_orders["_merge"].eq("both").all()

print("PASS: 행 수 유지, 미매칭 없음")
llm_orders = llm_orders.drop(columns="_merge")

# 5. 날짜 확인과 completed 필터
print("\n[5] 날짜 확인 및 completed 주문 선택")

llm_orders["order_date"] = pd.to_datetime(
    llm_orders["order_date"], errors="coerce"
)
llm_date_failures = int(llm_orders["order_date"].isna().sum())

print("날짜 변환 실패:", llm_date_failures)
assert llm_date_failures == 0

llm_completed = llm_orders.loc[
    llm_orders["order_status"] == "completed"
].copy()

assert not llm_completed.empty
assert llm_completed["order_status"].eq("completed").all()

llm_order_count = llm_completed["order_id"].nunique()

print("필터 전 금액:", f"{llm_orders['line_total'].sum():,}")
print("completed 금액:", f"{llm_completed['line_total'].sum():,}")
print("completed 상세 행 수:", len(llm_completed))
print("completed 고유 주문 수:", llm_order_count)

display(
    llm_completed[
        ["order_id", "order_status", "quantity", "unit_price", "line_total"]
    ].head()
)

# 6. 상품 카테고리 연결과 병합 검증
print("\n[6] products 병합과 검증")

print("products.product_id 결측:", products["product_id"].isna().sum())
print("products.product_id 중복:", products["product_id"].duplicated().sum())

assert products["product_id"].notna().all()
assert products["product_id"].is_unique

llm_sales = llm_completed.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

llm_products_counts = llm_sales["_merge"].value_counts()

print("병합 전 행 수:", len(llm_completed))
print("병합 후 행 수:", len(llm_sales))
display(llm_products_counts.rename("rows").to_frame())

assert len(llm_sales) == len(llm_completed)
assert llm_sales["_merge"].eq("both").all()
assert llm_sales["category"].notna().all()

print("PASS: 행 수 유지, 미매칭과 카테고리 결측 없음")
llm_sales = llm_sales.drop(columns="_merge")

llm_merge_checks = pd.DataFrame([
    {
        "merge": "order_items → orders",
        "before_rows": len(llm_items),
        "after_rows": len(llm_orders),
        "both": int(llm_orders_counts["both"]),
        "left_only": int(llm_orders_counts["left_only"]),
    },
    {
        "merge": "completed → products",
        "before_rows": len(llm_completed),
        "after_rows": len(llm_sales),
        "both": int(llm_products_counts["both"]),
        "left_only": int(llm_products_counts["left_only"]),
    },
])

display(llm_merge_checks)

# 7. 수정한 코드로 카테고리별 집계
print("\n[7] completed 주문의 카테고리별 집계")

llm_category_checked = (
    llm_sales.groupby("category", as_index=False, dropna=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
    .reset_index(drop=True)
)

display(llm_category_checked)

# 8. 원본 총합 및 앞선 분석 결과와 비교
print("\n[8] 수정 코드 결과 검증")

llm_completed_ids = orders.loc[
    orders["order_status"] == "completed", "order_id"
]

llm_original_total = llm_items.loc[
    llm_items["order_id"].isin(llm_completed_ids), "line_total"
].sum()

llm_total_checks = pd.DataFrame([
    {
        "scope": "원본 completed 상세",
        "total_sales": llm_original_total,
    },
    {
        "scope": "상품 병합 후 completed",
        "total_sales": llm_sales["line_total"].sum(),
    },
    {
        "scope": "수정 코드 카테고리 집계",
        "total_sales": llm_category_checked["total_sales"].sum(),
    },
    {
        "scope": "앞선 분석의 원본 합계",
        "total_sales": original_completed_total,
    },
])

llm_total_checks["difference"] = (
    llm_total_checks["total_sales"] - llm_original_total
)
llm_total_checks["result"] = (
    llm_total_checks["difference"]
    .eq(0)
    .map({True: "PASS", False: "FAIL"})
)

display(llm_total_checks)
assert llm_total_checks["difference"].eq(0).all()

assert (
    llm_category_checked["total_quantity"].sum()
    == llm_completed["quantity"].sum()
)

# 정렬 전 인덱스가 남아 있어도 실제 값을 비교하도록 양쪽을 정리한다.
pd.testing.assert_frame_equal(
    llm_category_checked
    .sort_values("category")
    .reset_index(drop=True),

    category_sales[["category", "total_quantity", "total_sales"]]
    .sort_values("category")
    .reset_index(drop=True),
)

assert llm_order_count == completed_order_sales["order_id"].nunique()

print("PASS: 원본과 수정 코드의 금액 총합 일치")
print("PASS: 기존 카테고리 집계와 수량·금액 일치")
print("PASS: 기존 분석과 고유 주문 수 일치")
print("최종 판단: 수정 후 사용")


[1] 실제 데이터 구조 확인
order_items: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
orders: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
products: ['product_id', 'product_name', 'category', 'price']


,order_count
order_status,
completed,184
cancelled,64
refunded,52



[2] 기존 LLM 코드 실행
order_items.groupby('category')['line_total'].sum()
오류: 'category'
category는 order_items가 아니라 products에 있다.
주문 상태도 orders에서 연결해야 completed만 선택할 수 있다.

[3] line_total = quantity × unit_price


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000



[4] orders 병합과 검증
orders.order_id 결측: 0
orders.order_id 중복: 0
병합 전 행 수: 764
병합 후 행 수: 764


...


[7] completed 주문의 카테고리별 집계


,category,total_quantity,total_sales
0,스포츠,295,31743000
1,전자기기,259,26400000
2,생활용품,272,23915000
3,뷰티,223,23383000
4,식품,133,16573000
5,도서,149,16389000
6,패션,111,10587000



[8] 수정 코드 결과 검증


,scope,total_sales,difference,result
0,원본 completed 상세,148990000,0,PASS
1,상품 병합 후 completed,148990000,0,PASS
2,수정 코드 카테고리 집계,148990000,0,PASS
3,앞선 분석의 원본 합계,148990000,0,PASS


PASS: 원본과 수정 코드의 금액 총합 일치
PASS: 기존 카테고리 집계와 수량·금액 일치
PASS: 기존 분석과 고유 주문 수 일치
최종 판단: 수정 후 사용


## 6. LLM pandas 코드 검증
- LLM Prompt 요약: 실습 Notebook의 집계 예시에서 실제 컬럼, completed 필터, 병합과 주문 수 계산, 총합 검증을 확인하는 요청을 정리했다. 코드 보완과 답안 작성에는 Codex의 도움을 받았다.
- 제안 코드 요약: order_items.groupby('category')['line_total'].sum().
- 실제 컬럼/범위와 맞지 않은 부분: order_items에는 category가 없었고 completed만 선택하는 조건도 없었다. 병합과 총합을 확인하는 코드도 빠져 있었다.
- 수정한 내용: 원본 order_items에서 line_total을 다시 계산하고 orders를 연결한 뒤 completed만 남겼다. products의 category를 연결할 때도 key 중복, 행 수, 미매칭을 확인했다. 날짜 변환 실패를 확인하고 주문 수는 order_id.nunique()로 계산했다. 아래 수정 코드의 총합과 카테고리별 값은 앞선 분석 결과와 같았다.
- 최종 판단: 수정 후 사용.

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단

원래 예시는 category가 없어서 KeyError가 났다.
이 컬럼만 연결한다고 끝나는 것도 아니었다. completed 필터가 없으면 취소·환불 주문까지 더해져 255,610,000이 나오기 때문이다.
상세 행 수 474개를 주문 수로 쓰는 것도 잘못된 계산이었다. 주문 ID를 중복 없이 세니 184건이었다.

수정 코드의 주문 병합은 764→764행, 상품 병합은 474→474행이었고 미매칭은 모두 0행이었다. 총합은 148,990,000, 고유 주문 수는 184건이었다.
카테고리 순서와 인덱스를 맞춘 뒤 수정한 집계가 category_sales와 같은지 비교했다. 점검 항목 8개도 모두 통과했다.
이 과정을 통해 LLM 코드도 실행 여부만 볼 것이 아니라, 내가 정한 질문과 범위에 맞게 계산하는지 확인해야 한다는 점을 알 수 있었다.

## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [31]:
# 과제 코드를 아래에 작성하세요.
import pandas as pd
from IPython.display import display

# 1. 40세 이상 고객 추출
print("\n[1] 40세 이상 고객")

customers_over_40 = customers.loc[
    customers["age"] >= 40,
    ["customer_id", "gender", "age", "city"],
].copy()

print("고객 수:", len(customers_over_40))
display(customers_over_40)


# 2. 상품 가격이 낮은 순서대로 10개 출력
print("\n[2] 가격이 낮은 상품 10개")

lowest_price_products = (
    products.sort_values(["price", "product_id"])
    .head(10)
)

display(lowest_price_products)


# 3. 결제수단별 주문 수와 비율 계산
print("\n[3] 결제수단별 주문 수와 비율 — 전체 주문 기준")

payment_summary = (
    orders.groupby("payment_method", as_index=False, dropna=False)
    .agg(order_count=("order_id", "nunique"))
    .sort_values("order_count", ascending=False)
    .reset_index(drop=True)
)

total_order_count = orders["order_id"].nunique()

payment_summary["order_ratio"] = (
    payment_summary["order_count"] / total_order_count * 100
).round(2)

display(payment_summary)
print("전체 고유 주문 수:", total_order_count)
print("order_ratio 단위: %")

assert payment_summary["order_count"].sum() == total_order_count


# 4. 카테고리별 평균 상품 가격 계산
print("\n[4] 카테고리별 평균 상품 가격 — 상품 목록의 단순 평균")

category_average_price = (
    products.groupby("category", as_index=False, dropna=False)
    .agg(average_price=("price", "mean"))
    .sort_values("average_price", ascending=False)
    .reset_index(drop=True)
)

category_average_price["average_price"] = (
    category_average_price["average_price"].round(2)
)

display(category_average_price)


# 5. 완료 주문과 전체 주문의 금액 차이 계산
print("\n[5] 전체 주문 금액과 completed 주문 금액 비교")

practice_items = order_items.copy()
practice_items["line_total"] = (
    practice_items["quantity"] * practice_items["unit_price"]
)

assert orders["order_id"].notna().all()
assert orders["order_id"].is_unique

practice_order_sales = practice_items.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("병합 전 행 수:", len(practice_items))
print("병합 후 행 수:", len(practice_order_sales))
display(
    practice_order_sales["_merge"]
    .value_counts()
    .rename("rows")
    .to_frame()
)

assert len(practice_order_sales) == len(practice_items)
assert practice_order_sales["_merge"].eq("both").all()

practice_order_sales = practice_order_sales.drop(columns="_merge")

practice_completed = practice_order_sales.loc[
    practice_order_sales["order_status"] == "completed"
].copy()

all_amount = practice_items["line_total"].sum()
completed_amount = practice_completed["line_total"].sum()
amount_difference = all_amount - completed_amount

amount_comparison = pd.DataFrame([
    {
        "scope": "전체 주문",
        "order_count": practice_order_sales["order_id"].nunique(),
        "total_amount": all_amount,
    },
    {
        "scope": "completed 주문",
        "order_count": practice_completed["order_id"].nunique(),
        "total_amount": completed_amount,
    },
])

display(amount_comparison)
print("금액 차이 — 전체에서 completed를 뺀 값:", f"{amount_difference:,}")

excluded_amount = practice_order_sales.loc[
    practice_order_sales["order_status"] != "completed",
    "line_total",
].sum()

assert practice_order_sales["line_total"].sum() == all_amount
assert amount_difference == excluded_amount

print("PASS: 금액 차이와 제외된 주문의 금액 합계가 같다.")
print("금액은 quantity × unit_price의 합계다.")


# 6. 고객별 구매 금액 코드 요청 프롬프트 작성
print("\n[6] 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트")

customer_sales_prompt = """
다음 데이터로 고객별 completed 주문 금액을 집계하는 pandas 코드를 작성해 주세요.

사용할 컬럼:
- order_items: order_item_id, order_id, quantity, unit_price
- orders: order_id, customer_id, order_status
- customers: customer_id, city

요구사항:
1. 실제 컬럼과 order_status 값을 먼저 확인해 주세요.
2. line_total = quantity * unit_price로 계산해 주세요.
3. orders.order_id의 결측치와 중복을 확인해 주세요.
4. order_items와 orders를 order_id로 left merge하고,
   validate="many_to_one", indicator=True를 적용해 주세요.
5. 병합 전후 행 수와 미매칭을 출력하고 검증해 주세요.
6. order_status == "completed"인 상세만 선택해 주세요.
7. customer_id별로 다음 값을 집계해 주세요.
   - total_sales: line_total의 합계
   - order_count: order_id.nunique()
8. customers.customer_id의 결측치와 중복을 확인한 뒤,
   집계 결과에 city를 left merge해 주세요.
   validate="one_to_one", indicator=True를 적용하고
   행 수와 미매칭을 검증해 주세요.
9. 상품·고객 정보를 병합하기 전 원본 상세에서
   completed 주문 ID에 해당하는 line_total 합계를 구하고,
   고객별 total_sales 합계와 같은지 assert로 검증해 주세요.
10. 결과는 total_sales가 큰 순서로 출력해 주세요.
    고객 이름은 출력하거나 저장하지 마세요.
11. 각 단계의 결과와 검증 과정이 출력으로 보이게 작성해 주세요.
    total_sales는 완료 주문 상세 금액의 합계이며,
    순매출이나 이익으로 해석하지 않도록 설명해 주세요.
"""

print(customer_sales_prompt)


[1] 40세 이상 고객
고객 수: 78


,customer_id,gender,age,city
2,3,F,61,성남
3,4,F,55,울산
6,7,F,53,인천
8,9,M,69,서울
9,10,F,62,울산
...,...,...,...,...
140,141,M,40,광주
142,143,F,43,대전
144,145,M,59,성남
145,146,M,61,성남



[2] 가격이 낮은 상품 10개


,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000



[3] 결제수단별 주문 수와 비율 — 전체 주문 기준


...

,category,average_price
0,식품,137142.86
1,뷰티,117687.50
2,패션,115909.09
3,스포츠,111578.95
4,도서,106857.14
5,전자기기,101588.24
6,생활용품,96437.50



[5] 전체 주문 금액과 completed 주문 금액 비교
병합 전 행 수: 764
병합 후 행 수: 764


,rows
_merge,
both,764
left_only,0
right_only,0


,scope,order_count,total_amount
0,전체 주문,300,255610000
1,completed 주문,184,148990000


금액 차이 — 전체에서 completed를 뺀 값: 106,620,000
PASS: 금액 차이와 제외된 주문의 금액 합계가 같다.
금액은 quantity × unit_price의 합계다.

[6] 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트

다음 데이터로 고객별 completed 주문 금액을 집계하는 pandas 코드를 작성해 주세요.

사용할 컬럼:
- order_items: order_item_id, order_id, quantity, unit_price
- orders: order_id, customer_id, order_status
- customers: customer_id, city

요구사항:
1. 실제 컬럼과 order_status 값을 먼저 확인해 주세요.
2. line_total = quantity * unit_price로 계산해 주세요.
3. orders.order_id의 결측치와 중복을 확인해 주세요.
4. order_items와 orders를 order_id로 left merge하고,
   validate="many_to_one", indicator=True를 적용해 주세요.
5. 병합 전후 행 수와 미매칭을 출력하고 검증해 주세요.
6. order_status == "completed"인 상세만 선택해 주세요.
7. customer_id별로 다음 값을 집계해 주세요.
   - total_sales: line_total의 합계
   - order_count: order_id.nunique()
8. customers.customer_id의 결측치와 중복을 확인한 뒤,
   집계 결과에 city를 left merge해 주세요.
   validate="one_to_one", indicator=True를 적용하고
   행 수와 미매칭을 검증해 주세요.
9. 상품·고객 정보를 병합하기 전 원본 상세에서
   completed 주문 ID에 해당하는 line_total 합계를 구하고,
   고객별 total_sales 합계와 같은지 a

## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. 완료 주문 금액이 가장 큰 카테고리는 스포츠였다. 금액은 31,743,000, 수량은 295개로 모두 가장 높았다. 전체 완료 주문 금액에서 차지하는 비율은 21.31%였다.
2. completed만 포함하는 조건이 결과에 큰 영향을 줬다. 취소·환불 주문을 빼기 전에는 255,610,000이었지만 빼고 나니 148,990,000이었다. 제외된 금액은 전체의 41.71%로, 주문 상태를 구분하지 않으면 결과를 크게 잘못 해석할 수 있었다.

### 그 결과를 뒷받침하는 수치/표

카테고리 집계표에서 스포츠의 금액과 수량이 가장 큰 것을 확인했다.
상태별 금액표에서는 cancelled 62,181,000과 refunded 44,439,000을 확인했다.
두 금액의 합은 전체와 completed의 차이인 106,620,000과 같았다. 집계별 총합도 모두 148,990,000으로 일치했다.

### 추가로 확인하고 싶은 질문

스포츠의 금액이 큰 이유를 더 살펴보고 싶다. 다른 카테고리보다 수량이 많아서인지, 단가가 높아서인지, 등록된 상품이 많아서인지 나눠 비교할 수 있을 것 같다.
카테고리마다 취소·환불 비율이 다른지도 확인하고 싶다.
원가 자료가 있다면 금액 순위와 실제 이익 순위가 같은지도 비교해 보고 싶다.

### 현재 결과의 한계

이번 결과는 가상 데이터에 포함된 기간의 주문만 설명한다.
원가나 재고, 상품 노출, 프로모션 자료가 없어 왜 더 많이 팔렸는지까지 알 수는 없다.
완료 주문이 없는 상품과 고객은 집계에 나오지 않았고, 고객 순위도 이 기간의 구매 금액만 비교한 결과다.
병합과 총합 검증을 통과했더라도 원본의 잘못된 입력까지 찾아낸 것은 아니다.

[Chapter 04 Evidence]

1. Notebook
- chapter04/chapter04.ipynb 실행 완료: 예

2. 병합 검증
- orders.order_id 중복: 0건
- orders merge 전 행 수: 764행
- orders merge 후 행 수: 764행
- orders merge 미매칭: 0행
- products merge 전 행 수: 474행
- products merge 후 행 수: 474행
- products merge 미매칭: 0행

3. 계산 범위
- 사용 주문 상태: completed
- line_total 계산식: quantity × unit_price
- 날짜 변환 실패: 0행

4. 교차 검증
- 완료 주문 상세 line_total 합계: 148,990,000
- category_sales 합계: 148,990,000
- monthly_summary 합계: 148,990,000
- customer_sales 합계: 148,990,000
- 합계 일치 여부: PASS

5. 저장 파일
- ch04_category_sales.csv: 존재
- ch04_product_sales.csv: 존재
- ch04_monthly_sales.csv: 존재
- ch04_customer_sales.csv: 존재

6. LLM 검증
- LLM이 빠뜨린 검증 항목: 실제 category 컬럼 위치 확인, completed 필터, 병합 key의 결측·중복 검사, validate와 indicator 적용, 병합 전후 행 수와 미매칭 확인, 원본 completed 금액과 집계 총합 비교.

7. 남은 질문
- 스포츠 카테고리의 완료 주문 금액이 가장 큰 이유는 판매 수량, 단가, 등록된 상품 수 중 무엇의 영향이 큰가?

## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.
